# ПЗ 5 — Детекция объектов YOLO

**Задача:** читать кадры из Drive, прогнать через YOLOv8, сохранить детекции в Drive.

**Стек:** `ultralytics` (YOLOv8), `opencv-python`, `pandas`

In [ ]:
!pip install ultralytics opencv-python-headless pandas tqdm -q

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE  = '/content/drive/MyDrive/cv-frames'
FRAMES_DIR  = f'{BASE_DRIVE}/кадры'
RESULTS_DIR = f'{BASE_DRIVE}/результаты'

os.makedirs(RESULTS_DIR, exist_ok=True)

frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))
print(f'✅ Найдено кадров: {len(frames)}')

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # nano — быстрая; s/m — точнее
print('✅ YOLO загружена')

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm

CONF = 0.4
rows = []

for fname in tqdm(frames, desc='YOLO'):
    res = model(f'{FRAMES_DIR}/{fname}', conf=CONF, verbose=False)
    for r in res:
        for box in r.boxes:
            rows.append({
                'frame':      fname,
                'frame_num':  int(fname.split('_')[1].split('.')[0]),
                'class':      model.names[int(box.cls[0])],
                'conf':       round(float(box.conf[0]), 3),
                'x1': round(float(box.xyxy[0][0])),
                'y1': round(float(box.xyxy[0][1])),
                'x2': round(float(box.xyxy[0][2])),
                'y2': round(float(box.xyxy[0][3])),
            })

df = pd.DataFrame(rows)
OUT = f'{RESULTS_DIR}/yolo_detections.csv'
df.to_csv(OUT, index=False)
print(f'✅ Детекций: {len(df)} → {OUT}')
df.head(10)

In [ ]:
# Статистика по классам
print(df.groupby('class')['conf'].agg(['count','mean']).sort_values('count', ascending=False))

In [ ]:
# Визуализация первого кадра
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

if frames:
    img = cv2.imread(f'{FRAMES_DIR}/{frames[0]}')
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(img_rgb)
    for _, row in df[df['frame'] == frames[0]].iterrows():
        rect = Rectangle((row.x1, row.y1), row.x2-row.x1, row.y2-row.y1,
                         linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        ax.text(row.x1, row.y1-5, f"{row['class']} {row.conf:.2f}", color='red', fontsize=9)
    ax.axis('off')
    plt.title(frames[0])
    plt.tight_layout()
    plt.show()